# Tracker evaluation

In [1]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src._config import DEFAULT_N_JOBS, DEFAULT_RESULTS_DIR

for _var in (
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
):
    os.environ[_var] = str(DEFAULT_N_JOBS)

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

ROOT = Path.cwd().parent
RESULTS_DIR = ROOT / DEFAULT_RESULTS_DIR / "eval_tracking/results/"
OUT_IMG = ROOT / "img/"
OUT_IMG.mkdir(parents=True, exist_ok=True)

plt.style.use(".matplotlib/paper.mplstyle")

# Toggle to `True` to save outputs
save_output = False

In [ ]:
VARIANT_ORDER = [
    "A_yolo_botsort",
    "A1_yolo_botsort_reid",
    "B_gs2_strict",
    "B_gs2_fixed",
    "C_sam3_frame_zero",
    "D_sam3_fixed",
    "E_sam3_adaptive",
]
VARIANT_LABELS = {
    "A_yolo_botsort": "YOLO26x + BoT-SORT (no re-ID)",
    "A1_yolo_botsort_reid": "YOLO26x + BoT-SORT (with re-ID)",
    "B_gs2_strict": "Grounded-SAM-2, frame-0 grounding, no recovery",
    "B_gs2_fixed": "Grounded-SAM-2, best-frame grounding + recovery",
    "C_sam3_frame_zero": "SAM 3, frame-0 grounding, fixed chunks",
    "D_sam3_fixed": "SAM 3, adaptive grounding, fixed chunks",
    "E_sam3_adaptive": "SAM 3, adaptive grounding + adaptive chunking",
}

VARIANT_COLORS = {
    # YOLO — teal family (light → dark)
    "A_yolo_botsort": "#88CCAA",
    "A1_yolo_botsort_reid": "#009E73",
    # GS2 — blue family (light → dark)
    "B_gs2_strict": "#7ECEF4",
    "B_gs2_fixed": "#0072B2",
    # SAM3 — orange/vermillion family (light → dark = best)
    "C_sam3_frame_zero": "#F5C49A",
    "D_sam3_fixed": "#E07B39",
    "E_sam3_adaptive": "#9C3D00",
}
STYLE_OVERRIDES = {
    "font.sans-serif": ["Arial", "Liberation Sans", "Arimo", "DejaVu Sans"],
    "axes.labelweight": "bold",
    "axes.axisbelow": True,
    "font.size": 12,
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
}

plt.rcParams.update(STYLE_OVERRIDES)


def grouped_legend(ax, **kwargs):
    """Add a method legend with separator lines between variant groups."""
    handles_all, labels_all = ax.get_legend_handles_labels()
    sep = lambda: Line2D([0, 1], [0, 0], color="lightgray", lw=1.0)
    legend = ax.legend(
        handles=[
            handles_all[0], handles_all[1], sep(),
            handles_all[2], handles_all[3], sep(),
            handles_all[4], handles_all[5], handles_all[6],
        ],
        labels=[
            labels_all[0], labels_all[1], "",
            labels_all[2], labels_all[3], "",
            labels_all[4], labels_all[5], labels_all[6],
        ],
        title="Method",
        **kwargs,
    )
    legend.get_title().set_fontweight("bold")
    for text in legend.get_texts():
        if text.get_text() == VARIANT_LABELS["E_sam3_adaptive"]:
            text.set_fontweight("bold")
    return legend

In [3]:
agg = (
    pd
    .read_csv(RESULTS_DIR / "metrics_aggregate.csv")
    .set_index("variant")
    .loc[VARIANT_ORDER]
)
agg = agg[["HOTA", "DetA", "AssA", "idf1", "mota"]]

per_video = pd.read_csv(RESULTS_DIR / "metrics_per_video.csv").set_index("variant")
sd = (
    per_video
    .groupby("variant")[["HOTA", "DetA", "AssA", "idf1", "mota"]]
    .std()
    .loc[VARIANT_ORDER]
)

In [4]:
agg.round(3)

,HOTA,DetA,AssA,idf1,mota
variant,,,,,
A_yolo_botsort,0.063,0.187,0.022,0.058,0.063
A1_yolo_botsort_reid,0.095,0.292,0.032,0.081,0.124
B_gs2_strict,0.202,0.172,0.243,0.246,-0.022
B_gs2_fixed,0.342,0.304,0.392,0.478,-0.375
C_sam3_frame_zero,0.283,0.398,0.208,0.363,0.394
D_sam3_fixed,0.539,0.637,0.470,0.669,0.661
E_sam3_adaptive,0.561,0.639,0.499,0.701,0.701


In [5]:
sd.round(3)

,HOTA,DetA,AssA,idf1,mota
variant,,,,,
A_yolo_botsort,0.016,0.047,0.006,0.022,0.030
A1_yolo_botsort_reid,0.024,0.054,0.013,0.040,0.042
B_gs2_strict,0.196,0.170,0.234,0.263,0.368
B_gs2_fixed,0.064,0.048,0.104,0.099,0.270
C_sam3_frame_zero,0.060,0.077,0.075,0.120,0.118
D_sam3_fixed,0.130,0.054,0.185,0.197,0.160
E_sam3_adaptive,0.057,0.032,0.110,0.116,0.072


# HOTA decomposition (AssA vs. DetA)

In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 7.0))

for v in VARIANT_ORDER:
    ax.errorbar(
        agg.loc[v, "DetA"],
        agg.loc[v, "AssA"],
        xerr=sd.loc[v, "DetA"],
        yerr=sd.loc[v, "AssA"],
        fmt="o",
        markersize=8,
        color=VARIANT_COLORS[v],
        markeredgecolor="white",
        markeredgewidth=1.5,
        ecolor=(*mcolors.to_rgb(VARIANT_COLORS[v]), 0.6),
        elinewidth=1.2,
        capsize=3,
        label=VARIANT_LABELS[v],
        zorder=3,
    )

xs = np.linspace(0.01, 0.85, 200)
for hota_iso in np.arange(0.1, 0.75, 0.1):
    ys = (hota_iso**2) / xs
    ax.plot(xs, ys, color="black", lw=0.8, ls="--", zorder=1)
    x_lab = 0.83
    y_lab = (hota_iso**2) / x_lab
    if 0.005 < y_lab < 0.83:
        ax.text(
            x_lab,
            y_lab,
            f"HOTA={hota_iso:.1f}",
            fontsize=10,
            color="black",
            va="bottom",
        )

ax.set_xlim(0, 0.85)
ax.set_ylim(0, 0.85)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("DetA (Detection Accuracy)")
ax.set_ylabel("AssA (Association Accuracy)")

grouped_legend(
    ax,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.13),
    ncol=1,
    handletextpad=0.6,
    borderaxespad=0.0,
)

fig.tight_layout()

if save_output:
    fig.savefig(OUT_IMG / "figS_tracker_eval_hota.pdf", dpi=300, bbox_inches="tight")
    fig.savefig(OUT_IMG / "figS_tracker_eval_hota.png", dpi=300, bbox_inches="tight")
plt.show()

# Barbraph all metrics

In [ ]:
W_PAGE = 6.5
SCALE = 1.0
W_FULL = W_PAGE * SCALE
H_MAX = 8.75 * SCALE

LEG_FONTSIZE = 8
AX_FONTSIZE = 10

fig, ax = plt.subplots(figsize=(W_FULL, 4.6 * SCALE))

metrics = ["HOTA", "DetA", "AssA", "idf1", "mota"]
x = np.arange(len(metrics))
n_variants = len(VARIANT_ORDER)
bar_width = 0.13


def bar_offset(i):
    return (i - (n_variants - 1) / 2) * bar_width


for i, v in enumerate(VARIANT_ORDER):
    vals = agg.loc[v, metrics].astype(float).values
    ax.bar(
        x + bar_offset(i),
        vals,
        bar_width,
        color=VARIANT_COLORS[v],
        edgecolor="#555",
        linewidth=0.3,
        label=VARIANT_LABELS[v],
        zorder=3,
    )

for i, v in enumerate(VARIANT_ORDER):
    vals = agg.loc[v, metrics].astype(float).values
    stds = sd.loc[v, metrics].astype(float).values
    ax.errorbar(
        x + bar_offset(i),
        vals,
        yerr=stds,
        fmt="none",
        ecolor="#222",
        elinewidth=1.0,
        capsize=2,
        zorder=4,
    )

# Dashed separator between HOTA family (HOTA, DetA, AssA) and independent metrics
ax.axvline(2.5, color="#aaa", lw=1.0, ls="--", zorder=1)

ax.yaxis.grid(True, color="#333", linewidth=0.5, alpha=0.4, zorder=0)
ax.axhline(0, color="#888", lw=0.8, zorder=2)
ax.set_xticks(x)
ax.set_xticklabels([""] * len(metrics))
ax.tick_params(axis="x", length=0)
ax.tick_params(axis="y", labelsize=9)
ax.set_ylabel("Score", fontsize=AX_FONTSIZE)
_lo = (agg[metrics] - sd[metrics]).min().min()
ax.set_ylim(_lo - 0.05, 1.0)

acronyms = ["HOTA", "DetA", "AssA", r"IDF$_1$", "MOTA"]
explainers = [
    "(Higher Order\nTracking Accuracy)",
    "(Detection\nAccuracy)",
    "(Association\nAccuracy)",
    "(ID F-Score)",
    "(Multi-Object\nTracking Accuracy)",
]
for xi, acr, exp in zip(x, acronyms, explainers):
    ax.text(
        xi,
        -0.04,
        acr,
        transform=ax.get_xaxis_transform(),
        ha="center",
        va="top",
        fontsize=9,
    )
    ax.text(
        xi,
        -0.12,
        exp,
        transform=ax.get_xaxis_transform(),
        ha="center",
        va="top",
        fontsize=7,
    )

legend = grouped_legend(
    ax,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.30),
    ncol=1,
    fontsize=LEG_FONTSIZE,
    handletextpad=0.6,
    labelspacing=0.35,
    handlelength=1.6,
    borderaxespad=0.0,
    borderpad=0.4,
)
legend.get_title().set_fontsize(LEG_FONTSIZE)

fig.subplots_adjust(left=0.105, right=0.985, top=0.97, bottom=0.48)

if save_output:
    fig.savefig(OUT_IMG / "figS_tracker_eval_metrics_agg.pdf", dpi=300)
    fig.savefig(OUT_IMG / "figS_tracker_eval_metrics_agg.png", dpi=300)
plt.show()